In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TestSpark") \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/05 18:15:54 WARN Utils: Your hostname, Kaylas-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 172.16.16.87 instead (on interface en0)
26/05/05 18:15:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 18:15:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
df = spark.read.csv("/Users/kaylajasmine/Documents/KULIAH/SEMESTER 6/BDA/HR-Employee-Attrition.csv", header=True, inferSchema=True)
df.show(5)

+---+---------+-----------------+---------+--------------------+----------------+---------+--------------+-------------+--------------+-----------------------+------+----------+--------------+--------+--------------------+---------------+-------------+-------------+-----------+------------------+------+--------+-----------------+-----------------+------------------------+-------------+----------------+-----------------+---------------------+---------------+--------------+------------------+-----------------------+--------------------+
|Age|Attrition|   BusinessTravel|DailyRate|          Department|DistanceFromHome|Education|EducationField|EmployeeCount|EmployeeNumber|EnvironmentSatisfaction|Gender|HourlyRate|JobInvolvement|JobLevel|             JobRole|JobSatisfaction|MaritalStatus|MonthlyIncome|MonthlyRate|NumCompaniesWorked|Over18|OverTime|PercentSalaryHike|PerformanceRating|RelationshipSatisfaction|StandardHours|StockOptionLevel|TotalWorkingYears|TrainingTimesLastYear|WorkLifeBalanc

26/05/05 18:15:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:
df = df.toDF(*[c.replace("ï»¿", "") for c in df.columns])

In [8]:
#rename target
df = df.withColumnRenamed("Attrition", "left_company")

In [9]:
#transformasi target ke numerik
from pyspark.sql.functions import col, when

df = df.withColumn(
    "left_company",
    when(col("left_company") == "Yes", 1).otherwise(0)
)

In [10]:
#drop kolom tidak penting
df = df.drop("EmployeeCount", "StandarHours", "Over18", "EmployeeNumber")

In [11]:
#filter data invalid
df = df.filter(col("Age").rlike("^[0-9]+$"))

In [12]:
#konversi tipe data
df = df.withColumn("Age", col("Age").cast("int"))
df = df.withColumn("MonthlyIncome", col("MonthlyIncome").cast("int"))
df = df.withColumn("JobSatisfaction", col("JobSatisfaction").cast("int"))
df = df.withColumn("PerformanceRating", col("PerformanceRating").cast("int"))
df = df.withColumn("WorkLifeBalance", col("WorkLifeBalance").cast("int"))
df = df.withColumn("YearsAtCompany", col("YearsAtCompany").cast("int"))

In [13]:
import numpy as np
np.__version__

'2.2.5'

In [14]:
from pyspark.ml.feature import StringIndexer

In [15]:
#FEATURE ENGINEERING -- encoding

categorical_cols = ["Department", "JobRole", "Gender", "MaritalStatus", "OverTime"]

for col_name in categorical_cols:
    indexer = StringIndexer(inputCol=col_name, outputCol=col_name + "_index")
    df = indexer.fit(df).transform(df)

In [20]:
#FEATURE ENGINEERING -- vector assembling
from pyspark.ml.feature import VectorAssembler

numeric_cols = [
    "Age", "MonthlyIncome", "JobSatisfaction",
    "PerformanceRating", "WorkLifeBalance",
    "YearsAtCompany"
]

assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="num_features"
)

df = assembler.transform(df)

In [22]:
#FEATURE ENGINEERING -- scaling
from pyspark.ml.feature import MinMaxScaler

scaler = MinMaxScaler(inputCol="num_features", outputCol="scaled_features")
df = scaler.fit(df).transform(df)

In [23]:
#FEATURE ENGINEERING -- split data
train, test = df.randomSplit([0.8, 0.2], seed=42)

In [24]:
#FINAL DATASET
df_final = df.select(
    "scaled_features",
    "Department_index",
    "JobRole_index",
    "Gender_index",
    "OverTime_index",
    "left_company"
)

df_final.show(5)

+--------------------+----------------+-------------+------------+--------------+------------+
|     scaled_features|Department_index|JobRole_index|Gender_index|OverTime_index|left_company|
+--------------------+----------------+-------------+------------+--------------+------------+
|[0.54761904761904...|             1.0|          0.0|         1.0|           1.0|           1|
|[0.73809523809523...|             0.0|          1.0|         0.0|           0.0|           0|
|[0.45238095238095...|             0.0|          2.0|         0.0|           1.0|           1|
|[0.35714285714285...|             0.0|          1.0|         1.0|           1.0|           0|
|[0.21428571428571...|             0.0|          2.0|         0.0|           0.0|           0|
+--------------------+----------------+-------------+------------+--------------+------------+
only showing top 5 rows
